In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory

# === 세션별 이력을 보관하는 메모리 저장소 ===
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

# === Chat Model ===
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

# === 어댑터즈 안내 챗봇 프롬프트 ===
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 어댑터즈 서비스 안내 담당자입니다. "
               "어댑터즈는 스타트업코드에서 운영하는 개발 교재 서빙 서비스이며, "
               "5단 분석법으로 교재를 작성합니다. 친절하게 답하세요."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# === 기본 체인 ===
chain = prompt | model | StrOutputParser()

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory

# === 체인 + 이력 관리 함수를 하나로 감싸기 ===
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [ ]:
config = {"configurable": {"session_id": "user-001"}}

response1 = chain_with_history.invoke(
    {"input": "어댑터즈는 어디에서 운영하는 서비스인가요?"},
    config=config
)
print(f"AI: {response1}")

In [ ]:
response2 = chain_with_history.invoke(
    {"input": "거기서 사용하는 교수법은 뭔가요?"},
    config=config
)
print(f"AI: {response2}")

In [ ]:
# === 다른 session_id 사용 ===
config_other = {"configurable": {"session_id": "user-002"}}

response_other = chain_with_history.invoke(
    {"input": "거기서 사용하는 교수법은 뭔가요?"},
    config=config_other
)
print(f"AI: {response_other}")